# Figure 5

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
            library(SeuratWrappers), 
                         
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ggplot2), 
            library(ggalluvial), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            library(circlize), 
            library(ggridges), 
            library(paletteer), 
            library(ComplexUpset), 
            library(egg), 
            library(ggh4x), 
            library(ComplexHeatmap), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.10.16-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()

In [7]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/line_plot.R")
source("script/figure/bin/dea_vp.R")
source("script/figure/bin/dea_stats.R")

In [8]:
options(repr.plot.width=5, repr.plot.height=10)

# Import Seurat

In [9]:
so <- readRDS("data/object/pp.rds")
so <- NormalizeData(so, assay="RNA", normalization.method="LogNormalize", scale.factor=1e4)

Normalizing layer: counts



# Import TCRdist results

In [68]:
cell_ir <- read.csv("result/ir/tcrdist/cell_ir.csv", row.names=1)

In [69]:
cell_ir$clone_id %>% unique() %>% length()
cell_ir$graph_clone_id %>% unique() %>% length()

[1] 8334

[1] 8251

In [70]:
# Add cell type annotation 
cell_ir <- cell_ir %>% tibble::rownames_to_column("cell_id") %>% dplyr::left_join(., so@meta.data %>% dplyr::select(cell_id, celltype_low), by=join_by(cell_id)) %>% tibble::column_to_rownames("cell_id")

In [71]:
# Make truncated patient ID
cell_ir <- cell_ir %>% dplyr::mutate(patient_id=gsub("atient_", "", patient_id))

In [72]:
# Remove invariant T cells
cell_ir <- cell_ir %>% dplyr::filter(celltype_low!="T['inv']")

# Alluvium plot clonal persistence 

In [73]:
plot_df <- cell_ir %>%
    dplyr::filter(time_point!="GVHD") %>% 
    dplyr::distinct(hto_demux_class, patient_id, clone_id, time_point) %>%
    dplyr::mutate(value=1) %>%
    tidyr::pivot_wider(names_from=time_point, values_from=value, values_fill=0) %>% 
    dplyr::mutate(across(c(Baseline, Tx, D14, D100), ~ if_else(.x == 1, "+", "-"))) 

In [74]:
plot_df <- plot_df %>%
    
    mutate(class=case_when(
    Baseline == "+" & (Tx=="+" | D14=="+" | D100=="+") ~ "persistent",
    Baseline == "+" & (Tx=="-" | D14=="-" | D100=="-")  ~ "lost",
    Baseline == "-" & (Tx=="+" | D14=="+" | D100=="+") ~ "gained",
    TRUE ~ "other"
    
    )
          )

In [75]:
plot_df$patient_id <- factor(plot_df$patient_id, levels=paste0("P", 1:7))

In [76]:
plot_df$class <- factor(plot_df$class, levels=rev(c("persistent", "gained", "lost", "other")))

In [77]:
plot_df$Baseline <- factor(plot_df$Baseline, levels = c("+", "-"))
plot_df$Tx <- factor(plot_df$Tx, levels = c("+", "-"))
plot_df$D14 <- factor(plot_df$D14, levels = c("+", "-"))
plot_df$D100 <- factor(plot_df$D100,levels = c("+", "-"))

In [78]:
# Store persistent clone IDs
clone_id_persistent <- plot_df %>% dplyr::filter(class=="persistent") %>% dplyr::pull(clone_id) %>% unique()
clone_id_gained <- plot_df %>% dplyr::filter(class=="gained") %>% dplyr::pull(clone_id) %>% unique()
clone_id_lost <- plot_df %>% dplyr::filter(class=="lost") %>% dplyr::pull(clone_id) %>% unique()

In [79]:
p_1 <- ggplot(plot_df %>% dplyr::filter(hto_demux_class=="Blood"), aes(axis1=Baseline, axis2=Tx, axis3=D14, axis4=D100)) +

    geom_alluvium(aes(fill=class), width=0.2, alpha=1.0) +
    geom_stratum(width=0.2, fill="white", color="black", size=0.4) +
    geom_text(stat="stratum", aes(label=after_stat(stratum)), size=2) +
    labs(x=NULL, y=paste("Number of", "clones", "groups")) +
    scale_fill_manual(values=c(
        
        "persistent"="#7393B3",
        "lost"="#808080",
        "gained"="#B2D4B6",
        "other"="#000000"
    
    )
                     ) + 

    facet_grid(patient_id~hto_demux_class, scales="free", space="free") + 
    theme_global_set(4) + theme(legend.position="none")

In [80]:
p_2 <- ggplot(plot_df %>% dplyr::filter(hto_demux_class=="Skin"), aes(axis1=Baseline, axis2=Tx, axis3=D14, axis4=D100)) +

    geom_alluvium(aes(fill=class), width=0.2, alpha=1.0) +
    geom_stratum(width=0.2, fill="white", color="black", size=0.4) +
    geom_text(stat="stratum", aes(label=after_stat(stratum)), size=2) +
    labs(x=NULL, y=paste("Number of", "clones", "groups")) +
    scale_fill_manual(values=c(
        
        "persistent"="#7393B3",
        "lost"="#808080",
        "gained"="#B2D4B6",
        "other"="#000000"
    
    )
                     ) + 

    facet_grid(patient_id~hto_demux_class, scales="free", space="free") + 
    theme_global_set(4) + theme(legend.position="none")

In [81]:
pdf("result/figures/figure_5/alluvium_clone.pdf", width=10, height=30)

gridExtra::grid.arrange(

    p_1 %>% egg::set_panel_size(., width=unit(2.5, "cm"), height=unit(1, "cm")), 
    p_2 %>% egg::set_panel_size(., width=unit(2.5, "cm"), height=unit(1, "cm")), ncol=2, nrow=1

) 

dev.off()

pdf 
  2

# Pie Charts of persitent clones per cell type

In [82]:
data <- cell_ir %>% 

    dplyr::filter(time_point!="GVHD") %>%

    dplyr::filter(clone_id %in% clone_id_persistent) %>% 
    dplyr::select(hto_demux_class, celltype_low, clone_id) %>% 

    dplyr::distinct() %>% 

    group_by(hto_demux_class, celltype_low) %>% 
    dplyr::summarise(count=n(), .groups="drop") %>% dplyr::mutate(group=celltype_low) %>% 
    dplyr::mutate(group=factor(group, levels=names(color$celltype_low_r)))

In [83]:
cell_ir %>% 
    dplyr::filter(time_point!="GVHD") %>%
    dplyr::filter(clone_id %in% clone_id_persistent) %>% 
    dplyr::select(hto_demux_class, clone_id) %>% dplyr::distinct() %>% dplyr::group_by(hto_demux_class) %>% dplyr::summarise(n=n())

hto_demux_class,n
<chr>,<int>
Blood,46
Skin,91


In [84]:
p_1 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Blood"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

p_2 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Skin"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

In [85]:
data <- cell_ir %>% 

    dplyr::filter(time_point!="GVHD") %>%

    dplyr::filter(clone_id %in% clone_id_gained) %>% 
    dplyr::select(hto_demux_class, celltype_low, clone_id) %>% 

    dplyr::distinct() %>% 
    group_by(hto_demux_class, celltype_low) %>% 
    dplyr::summarise(count=n(), .groups="drop") %>% dplyr::rename(group=celltype_low) %>% 
    dplyr::mutate(group=factor(group, levels=names(color$celltype_low_r)))

In [86]:
cell_ir %>% 
    dplyr::filter(time_point!="GVHD") %>%
    dplyr::filter(clone_id %in% clone_id_gained) %>% 
    dplyr::select(hto_demux_class, clone_id) %>% dplyr::distinct() %>% dplyr::group_by(hto_demux_class) %>% dplyr::summarise(n=n())

hto_demux_class,n
<chr>,<int>
Blood,2536
Skin,447


In [87]:
p_3 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Blood"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

p_4 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Skin"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

In [88]:
data <- cell_ir %>% 

    dplyr::filter(time_point!="GVHD") %>%

    dplyr::filter(clone_id %in% clone_id_lost) %>% 
    dplyr::select(hto_demux_class, celltype_low, clone_id) %>% 

    dplyr::distinct() %>% 
    group_by(hto_demux_class, celltype_low) %>% 
    dplyr::summarise(count=n(), .groups="drop") %>% dplyr::rename(group=celltype_low) %>% 
    dplyr::mutate(group=factor(group, levels=names(color$celltype_low_r)))

In [89]:
cell_ir %>% 
    dplyr::filter(time_point!="GVHD") %>%
    dplyr::filter(clone_id %in% clone_id_lost) %>% 
    dplyr::select(hto_demux_class, clone_id) %>% dplyr::distinct() %>% dplyr::group_by(hto_demux_class) %>% dplyr::summarise(n=n())

hto_demux_class,n
<chr>,<int>
Blood,3417
Skin,769


In [90]:
p_5 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Blood"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

p_6 <- ggplot(data %>% dplyr::filter(hto_demux_class=="Skin"), aes(x="", y=count, fill=group)) +
    geom_col(width=1) +
    coord_polar(theta="y") + 
    scale_fill_manual(values=color$celltype_low_r) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0.25, keyheight=0.25, default.unit="cm")) 

In [91]:
pdf("result/figures/figure_5/pichart_clone_celltype_low_persistent.pdf", width=15, height=10)

gridExtra::grid.arrange(

    p_1 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    p_2 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    p_3 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    p_4 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    p_5 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    p_6 %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(1.5, "cm")), 
    
    
    
    ncol=6, nrow=1

) 

dev.off()

pdf 
  2

# Upset plots

## Clone ID

In [92]:
data_1 <- cell_ir %>% dplyr::distinct(clone_id, patient_id) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=patient_id, values_from=value, values_fill=0) %>% as.data.frame()
data_2 <- cell_ir %>% dplyr::distinct(clone_id, hto_demux_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=hto_demux_class, values_from=value, values_fill=0) %>% as.data.frame()
data_3 <- cell_ir %>% dplyr::distinct(clone_id, time_point) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=time_point, values_from=value, values_fill=0) %>% as.data.frame()
data_4 <- cell_ir %>% dplyr::distinct(clone_id, genotype_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=genotype_class, values_from=value, values_fill=0) %>% as.data.frame()
data_5 <- cell_ir %>% dplyr::distinct(clone_id, celltype_low) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=celltype_low, values_from=value, values_fill=0) %>% as.data.frame()

In [93]:
intersection_pp <- function(data) {

    sets <- colnames(data)[-1]
    
    data %>% count(across(all_of(sets)), name="intersection_size") %>% 
        dplyr::mutate(group=row_number(), pattern=apply(across(all_of(sets)), 1, paste0, collapse="")) %>%
        dplyr::arrange(desc(intersection_size)) %>%
        dplyr::mutate(group=row_number())

    
}

In [94]:
intersection_pp_1 <- intersection_pp(data_1)
intersection_pp_2 <- intersection_pp(data_2)
intersection_pp_3 <- intersection_pp(data_3)
intersection_pp_4 <- intersection_pp(data_4)
intersection_pp_5 <- intersection_pp(data_5)

In [95]:
intersection_pl <- function(x) {
    
    ggplot(x, aes(x=factor(group), y=intersection_size)) + 
        geom_col(width=0.8, linewidth=0.4, fill="white", color="black") + 
        scale_y_continuous(expand=c(0, 0)) + 
        xlab(NULL) + ylab("Clone [count]") + 
        theme_global_set(4) +
        theme(axis.text.x=element_blank(), axis.ticks.x=element_blank())


}

In [96]:
intersection_pl_1 <- intersection_pl(intersection_pp_1)
intersection_pl_2 <- intersection_pl(intersection_pp_2)
intersection_pl_3 <- intersection_pl(intersection_pp_3)
intersection_pl_4 <- intersection_pl(intersection_pp_4)
intersection_pl_5 <- intersection_pl(intersection_pp_5)

In [97]:
matrix_pp <- function(data, intersection) {

    sets <- colnames(data)[-1]
    
    intersection %>% 
      dplyr::select(group, all_of(sets)) %>% 
      tidyr::pivot_longer(cols=all_of(sets), names_to="set", values_to="present") %>%
      dplyr::mutate(set=factor(set, levels=rev(sets)), group=factor(group, levels=intersection$group))

    
}

In [98]:
matrix_pp_1 <- matrix_pp(data_1, intersection_pp_1)
matrix_pp_2 <- matrix_pp(data_2, intersection_pp_2)
matrix_pp_3 <- matrix_pp(data_3, intersection_pp_3)
matrix_pp_4 <- matrix_pp(data_4, intersection_pp_4)
matrix_pp_5 <- matrix_pp(data_5, intersection_pp_5)

In [99]:
matrix_pp_1 <- matrix_pp_1 %>% dplyr::mutate(set=factor(set, levels=rev(paste0("P", 1:12))))
matrix_pp_2 <- matrix_pp_2 %>% dplyr::mutate(set=factor(set, levels=rev(c("Blood", "Skin"))))
matrix_pp_3 <- matrix_pp_3 %>% dplyr::mutate(set=factor(set, levels=rev(c("Baseline", "Tx", "D14", "D100", "GVHD"))))
matrix_pp_4 <- matrix_pp_4 %>% dplyr::mutate(set=factor(set, levels=rev(c("Host", "Donor"))))

In [100]:
matrix_pl <- function(x, y) {

    label_map <- y %>% dplyr::select(intersection_size, group)

    ggplot(x, aes(x=group, y=set)) +
        geom_point(size=1.5, color="black", fill="white", shape=21) +
        geom_point(data=~ dplyr::filter(.x, present == 1), size=1.5, color="black", fill="black", shape=21) +
        geom_line(data=~ dplyr::filter(.x, present == 1), aes(group=group), color="black") +
        scale_x_discrete(labels=setNames(label_map$intersection_size, label_map$group)) +
        xlab(NULL) + ylab(NULL) + 
        theme_global_set(4) + theme(axis.text.x=element_text(angle=90, hjust=1, vjust=0.5))
    
}

In [101]:
matrix_pl_1 <- matrix_pl(matrix_pp_1, intersection_pp_1)
matrix_pl_2 <- matrix_pl(matrix_pp_2, intersection_pp_2)
matrix_pl_3 <- matrix_pl(matrix_pp_3, intersection_pp_3)
matrix_pl_4 <- matrix_pl(matrix_pp_4, intersection_pp_4)
matrix_pl_5 <- matrix_pl(matrix_pp_5, intersection_pp_5)

In [102]:
pdf("result/figures/figure_5/upset_clone.pdf", width=20, height=2.5)

gridExtra::grid.arrange(

    intersection_pl_1 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_2 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_3 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_4 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")),
    intersection_pl_5 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")),
    
    matrix_pl_1 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(.$data$set %>% unique() %>% length() / 5, "cm")), 
    matrix_pl_2 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(.$data$set %>% unique() %>% length() / 5, "cm")), 
    matrix_pl_3 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(.$data$set %>% unique() %>% length() / 5, "cm")), 
    matrix_pl_4 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(.$data$set %>% unique() %>% length() / 5, "cm")), 
    matrix_pl_5 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(.$data$set %>% unique() %>% length() / 5, "cm")), ncol=5, nrow=2

) 

dev.off()

pdf 
  2

## Clone ID

In [103]:
data_1 <- cell_ir %>% dplyr::distinct(graph_clone_id, patient_id) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=patient_id, values_from=value, values_fill=0)
data_2 <- cell_ir %>% dplyr::distinct(graph_clone_id, hto_demux_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=hto_demux_class, values_from=value, values_fill=0)
data_3 <- cell_ir %>% dplyr::distinct(graph_clone_id, time_point) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=time_point, values_from=value, values_fill=0)
data_4 <- cell_ir %>% dplyr::distinct(graph_clone_id, genotype_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=genotype_class, values_from=value, values_fill=0)
data_5 <- cell_ir %>% dplyr::distinct(graph_clone_id, celltype_low) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=celltype_low, values_from=value, values_fill=0)

In [104]:
intersection_pp <- function(data) {

    sets <- colnames(data)[-1]
    
    data %>% count(across(all_of(sets)), name="intersection_size") %>% 
        dplyr::mutate(group=row_number(), pattern=apply(across(all_of(sets)), 1, paste0, collapse="")) %>%
        dplyr::arrange(desc(intersection_size)) %>%
        dplyr::mutate(group=row_number())

    
}

In [105]:
intersection_pp_1 <- intersection_pp(data_1)
intersection_pp_2 <- intersection_pp(data_2)
intersection_pp_3 <- intersection_pp(data_3)
intersection_pp_4 <- intersection_pp(data_4)
intersection_pp_5 <- intersection_pp(data_5)

In [106]:
intersection_pl <- function(x) {
    
    ggplot(x, aes(x=factor(group), y=intersection_size)) + 
        geom_col(width=0.8, linewidth=0.4, fill="white", color="black") + 
        scale_y_continuous(expand=c(0, 0)) + 
        xlab(NULL) + ylab("Clone [count]") + 
        theme_global_set(4) +
        theme(axis.text.x=element_blank(), axis.ticks.x=element_blank())


}

In [107]:
intersection_pl_1 <- intersection_pl(intersection_pp_1)
intersection_pl_2 <- intersection_pl(intersection_pp_2)
intersection_pl_3 <- intersection_pl(intersection_pp_3)
intersection_pl_4 <- intersection_pl(intersection_pp_4)
intersection_pl_5 <- intersection_pl(intersection_pp_5)

In [108]:
matrix_pp <- function(data, intersection) {

    sets <- colnames(data)[-1]
    
    intersection %>% 
      dplyr::select(group, all_of(sets)) %>% 
      tidyr::pivot_longer(cols=all_of(sets), names_to="set", values_to="present") %>%
      dplyr::mutate(set=factor(set, levels=rev(sets)), group=factor(group, levels=intersection$group))

    
}

In [109]:
matrix_pp_1 <- matrix_pp(data_1, intersection_pp_1)
matrix_pp_2 <- matrix_pp(data_2, intersection_pp_2)
matrix_pp_3 <- matrix_pp(data_3, intersection_pp_3)
matrix_pp_4 <- matrix_pp(data_4, intersection_pp_4)
matrix_pp_5 <- matrix_pp(data_5, intersection_pp_5)

In [110]:
matrix_pp_1 <- matrix_pp_1 %>% dplyr::mutate(set=factor(set, levels=rev(paste0("P", 1:12))))
matrix_pp_2 <- matrix_pp_2 %>% dplyr::mutate(set=factor(set, levels=rev(c("Blood", "Skin"))))
matrix_pp_3 <- matrix_pp_3 %>% dplyr::mutate(set=factor(set, levels=rev(c("Baseline", "Tx", "D14", "D100", "GVHD"))))
matrix_pp_4 <- matrix_pp_4 %>% dplyr::mutate(set=factor(set, levels=rev(c("Host", "Donor"))))

In [111]:
matrix_pl <- function(x, y) {

    label_map <- y %>% dplyr::select(intersection_size, group)

    ggplot(x, aes(x=group, y=set)) +
        geom_point(size=1.5, color="black", fill="white", shape=21) +
        geom_point(data=~ dplyr::filter(.x, present == 1), size=1.5, color="black", fill="black", shape=21) +
        geom_line(data=~ dplyr::filter(.x, present == 1), aes(group=group), color="black") +
        scale_x_discrete(labels=setNames(label_map$intersection_size, label_map$group)) +
        xlab(NULL) + ylab(NULL) + 
        theme_global_set(4) + theme(axis.text.x=element_text(angle=90, hjust=1, vjust=0.5))
    
}

In [112]:
matrix_pl_1 <- matrix_pl(matrix_pp_1, intersection_pp_1)
matrix_pl_2 <- matrix_pl(matrix_pp_2, intersection_pp_2)
matrix_pl_3 <- matrix_pl(matrix_pp_3, intersection_pp_3)
matrix_pl_4 <- matrix_pl(matrix_pp_4, intersection_pp_4)
matrix_pl_5 <- matrix_pl(matrix_pp_5, intersection_pp_5)

In [113]:
pdf("result/figures/figure_5/upset_graph_clone.pdf", width=20, height=2.5)

gridExtra::grid.arrange(

    intersection_pl_1 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_2 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_3 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")), 
    intersection_pl_4 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")),
    intersection_pl_5 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length() / 5, "cm"), height=unit(1, "cm")),
    
    matrix_pl_1 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length()  / 5, "cm"), height=unit(.$data$set %>% unique() %>% length()  / 5, "cm")), 
    matrix_pl_2 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length()  / 5, "cm"), height=unit(.$data$set %>% unique() %>% length()  / 5, "cm")), 
    matrix_pl_3 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length()  / 5, "cm"), height=unit(.$data$set %>% unique() %>% length()  / 5, "cm")), 
    matrix_pl_4 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length()  / 5, "cm"), height=unit(.$data$set %>% unique() %>% length()  / 5, "cm")), 
    matrix_pl_5 %>% egg::set_panel_size(., width=unit(.$data$group %>% unique() %>% length()  / 5, "cm"), height=unit(.$data$set %>% unique() %>% length()  / 5, "cm")), ncol=5, nrow=2

) 

dev.off()

pdf 
  2

# Clonal expansion 

In [114]:
clone_id_1 <- cell_ir %>% dplyr::distinct(clone_id, hto_demux_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=hto_demux_class, values_from=value, values_fill=0) %>% as.data.frame() %>% tibble::column_to_rownames("clone_id") %>% dplyr::filter(rowSums(.)>1) %>% rownames() %>% unique()
clone_id_2 <- cell_ir %>% dplyr::distinct(clone_id, time_point) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=time_point, values_from=value, values_fill=0) %>% as.data.frame() %>% tibble::column_to_rownames("clone_id") %>% dplyr::filter(rowSums(.)>1) %>% rownames() %>% unique()

In [115]:
length(clone_id_1)

[1] 145

In [116]:
length(clone_id_2)

[1] 143

In [117]:
table(clone_id_1 %in% clone_id_2)


FALSE  TRUE 
  114    31 

In [118]:
clone_id_1 <- cell_ir %>% dplyr::distinct(graph_clone_id, hto_demux_class) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=hto_demux_class, values_from=value, values_fill=0) %>% as.data.frame() %>% tibble::column_to_rownames("graph_clone_id") %>% dplyr::filter(rowSums(.)>1) %>% rownames() %>% unique()
clone_id_2 <- cell_ir %>% dplyr::distinct(graph_clone_id, time_point) %>% mutate(value=1) %>% tidyr::pivot_wider(names_from=time_point, values_from=value, values_fill=0) %>% as.data.frame() %>% tibble::column_to_rownames("graph_clone_id") %>% dplyr::filter(rowSums(.)>1) %>% rownames() %>% unique()

In [119]:
length(clone_id_1)

[1] 145

In [120]:
length(clone_id_2)

[1] 152